In [1]:
# 0

import warnings
from pathlib import Path
from itertools import combinations

import pandas as pd
import numpy as np

from scipy import stats
import statsmodels.formula.api as smf

from sklearn.linear_model import lasso_path
from sklearn.exceptions import ConvergenceWarning

from joblib import Parallel, delayed

warnings.filterwarnings('ignore', category=ConvergenceWarning)
warnings.filterwarnings('ignore', category=FutureWarning)
warnings.filterwarnings('ignore', category=RuntimeWarning)

In [2]:
# 2

FVOL_COLS = ['actq', 'rectq', 'invtq', 'ppentq', 'atq', 'lctq', 'dlcq',
             'apq', 'dlttq', 'ltq', 'seqq', 'xoprq', 'cogsq', 'xsgaq']

def compute_fvol(df, deflator='saleq', fvol_cols=FVOL_COLS, diff_lag=4, win=8, min_obs=4):
    d = df.sort_values(['code', 'date']).reset_index(drop=True).copy()
    d['xoprq'] = d['cogsq'] + d['xsgaq']

    pq = pd.PeriodIndex(d['date'], freq='Q')
    d['qidx'] = pq.year * 4 + (pq.quarter - 1)

    g = d.groupby('code', sort=False)
    has_lag = (d['qidx'] - g['qidx'].shift(diff_lag)) == diff_lag
    for c in fvol_cols:
        d[f'd_{c}'] = np.where(has_lag, d[c] - g[c].shift(diff_lag), np.nan)

    g = d.groupby('code', sort=False)
    for c in fvol_cols:
        d[f'std_{c}'] = (g[f'd_{c}'].apply(lambda s: s.rolling(win, min_periods=min_obs).std())
                         .reset_index(level=0, drop=True))

    d['defl_pos'] = d[deflator].where(d[deflator] > 0)
    g = d.groupby('code', sort=False)
    d['avg_defl'] = (g['defl_pos'].apply(lambda s: s.rolling(win, min_periods=min_obs).mean())
                     .reset_index(level=0, drop=True))
    d['avg_defl'] = d['avg_defl'].where(d['avg_defl'] > 0)

    ind = []
    for c in fvol_cols:
        d[f'fvol_{c}'] = d[f'std_{c}'] / d['avg_defl']
        ind.append(f'fvol_{c}')
    return d, ind

def finalize_fvol(d, ind, fvol_cols=FVOL_COLS, min_valid=10, suffix=''):
    d = d.copy()
    rcols = [f'rank_{c}' for c in fvol_cols]
    ranked = d.groupby('date')[ind].rank(pct=True)
    ranked.columns = rcols
    d = pd.concat([d, ranked], axis=1)
    d['n_valid'] = d[rcols].notna().sum(axis=1)
    col = f'FVOL{suffix}'
    d[col] = d[rcols].mean(axis=1)
    d.loc[d['n_valid'] < min_valid, col] = np.nan
    return d, col

financials = pd.read_parquet('financials.parquet')

fv_sale, ind = compute_fvol(financials, deflator='saleq')
fv_sale, col_s = finalize_fvol(fv_sale, ind, suffix='_sale')

fv_asset, ind = compute_fvol(financials, deflator='atq')
fv_asset, col_a = finalize_fvol(fv_asset, ind, suffix='_asset')

fvol = fv_sale[['code', 'date', col_s]].merge(
    fv_asset[['code', 'date', col_a]], on=['code', 'date'])

for c in [col_s, col_a]:
    v = fvol[fvol[c].notna()]
    print(f'{c}: valid {len(v)}  avg firms/q {v.groupby("date").size().mean():.0f}  '
          f'range {v["date"].min().date()}~{v["date"].max().date()}')
    print(f'  describe: {v[c].describe()[["mean","std","min","50%","max"]].round(3).to_dict()}')

corr = fvol[[col_s, col_a]].corr().iloc[0, 1]
print(f'매출분모 FVOL vs 자산분모 FVOL 상관: {corr:.3f}')
print('분기당 종목수 (매출분모):')
gq = fvol[fvol[col_s].notna()].groupby('date').size()
display(gq.iloc[[0, len(gq)//4, len(gq)//2, 3*len(gq)//4, -1]]
        .rename_axis('date').reset_index(name='종목수'))

FVOL_sale: valid 66765  avg firms/q 681  range 2001-12-31~2026-03-31
  describe: {'mean': 0.501, 'std': 0.203, 'min': 0.021, '50%': 0.483, 'max': 1.0}
FVOL_asset: valid 66828  avg firms/q 682  range 2001-12-31~2026-03-31
  describe: {'mean': 0.501, 'std': 0.189, 'min': 0.022, '50%': 0.49, 'max': 1.0}
매출분모 FVOL vs 자산분모 FVOL 상관: 0.713
분기당 종목수 (매출분모):


,date,종목수
0,2001-12-31,557
1,2007-12-31,639
2,2014-03-31,675
3,2020-03-31,729
4,2026-03-31,763


In [ ]:
# 진단

delisted = pd.read_parquet('delisted.parquet')
financials = pd.read_parquet('financials.parquet')

fin_has = financials.groupby('code')['atq'].apply(lambda s: s.notna().any())
fin_codes = set(fin_has[fin_has].index)
d = delisted[delisted['code'].isin(fin_codes)].copy()

print(f'재무 존재 폐지 종목: {len(d)}개')

print('=== 1. reason_cat별 종목 수 ===')
display(d['reason_cat'].value_counts().rename_axis('reason_cat').reset_index(name='종목수'))

print('=== 2. reason 원문 전체 (reason_cat별) ===')
for cat in d['reason_cat'].value_counts().index:
    sub = d[d['reason_cat'] == cat]
    print(f'--- {cat} ({len(sub)}개) ---')
    display(sub['reason'].value_counts().rename_axis('reason').reset_index(name='n'))

print('=== 3. reason 원문 고유값 전체 목록 ===')
for r in sorted(d['reason'].unique()):
    print(repr(r))

print('=== 4. 사유별 폐지 연도 분포 ===')
d['year'] = d['del_date'].dt.year
display(pd.crosstab(d['year'], d['reason_cat']))